In [141]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
import sys

df = pd.read_csv("data/forestfires.csv")
y_label = 'area_bin'
df

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.00
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.00
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.00
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.00
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,6.44
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,54.29
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,11.16
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,0.00


In [76]:
df_pre = df.copy()
df_pre['month'] = df_pre['month'].astype('category').cat.codes
df_pre['day'] = df_pre['day'].astype('category').cat.codes
df_pre['area'] = ma.log(df_pre['area'].values).filled(0)
area_labels = [1, 2, 3, 4, 5, 6]
df_pre['area_bin'] = pd.cut(df_pre['area'], include_lowest=True, bins=6, labels=area_labels)
df_pre.drop(columns=['area'], inplace=True)
df_pre

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area_bin
0,7,5,7,0,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,2
1,7,4,10,5,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,2
2,7,4,10,2,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,2
3,8,6,7,0,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,2
4,8,6,7,3,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,1,3,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0,3
513,2,4,1,3,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0,5
514,7,4,1,3,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0,4
515,1,4,1,2,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0,2


In [79]:
df_x = df_pre.drop(columns=[y_label])
df_y = df_pre[y_label]
x_train, x_test, y_train, y_test = train_test_split(df_x, df_y, test_size=0.3, random_state=1)
y_test.value_counts()

2    95
3    31
4    19
5     7
1     3
6     1
Name: area_bin, dtype: int64

In [ ]:
def get_sensitivity(cm, label):
    labels = [1,2,3,4,5,6]
    #cm = confusion_matrix(y_test, y_pred, labels=labels)
    #print("{}/{}".format(cm[label - 1, label - 1], sum(cm[label - 1])))
    return cm[label - 1, label - 1] / sum(cm[label - 1])

def get_specificity(cm, label):
    indexes = [0,1,2,3,4,5]
    #cm = confusion_matrix(y_test, y_pred, labels=[1,2,3,4,5,6])
    tn = sum(np.diag(cm)) - cm[label - 1, label - 1]
    fp = sum([cm[i][label - 1] for i in indexes]) - cm[label - 1, label - 1]
    #print("tn:{}, fp:{}".format(tn, fp))
    return tn / (tn + fp)

def display_sens_spec(y_pred, y_test):
    labels = [1,2,3,4,5,6]
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    result = pd.DataFrame(columns=['label', 'sensitivity', 'specificity'])
    for idx, label in enumerate(labels):
        result.loc[idx] = [label, get_sensitivity(cm, label), get_specificity(cm, label)]
    result.set_index('label', inplace=True)
    return result    

In [122]:
dt_clf = DecisionTreeClassifier(criterion='gini', max_depth=3)
dt_clf.fit(x_train, y_train)
y_pred = dt_clf.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, labels=[1,2,3,4,5,6]))
print(display_sens_spec(y_pred, y_test))

Accuracy: 0.6089743589743589
[[ 0  3  0  0  0  0]
 [ 0 95  0  0  0  0]
 [ 0 28  0  3  0  0]
 [ 0 19  0  0  0  0]
 [ 0  7  0  0  0  0]
 [ 0  1  0  0  0  0]]
       sensitivity  specificity
label                          
1.0            0.0     1.000000
2.0            1.0     0.000000
3.0            0.0     1.000000
4.0            0.0     0.969388
5.0            0.0     1.000000
6.0            0.0     1.000000


In [139]:
def dict_product(d):
    keys = d.keys()
    for element in product(*d.values()):
        yield dict(zip(keys, element))
        
def GridSearchWithVal(model_class, param_grid, metrics='accuracy', cv=5, x_input=None, y_input=None):
    combinations = list(dict_product(param_grid))
    max_metrics = 0
    best_comb = None
    X = x_input if x_input is not None else x_train
    Y = y_input if y_input is not None else y_train
    print("{} combinations in total. Metric: {}".format(len(combinations), metrics))
    for idx, comb in enumerate(combinations):
        model = model_class(**comb)
        model.fit(x_train, y_train)
        y_preds = model.predict(x_test)
        # print("y_preds:{}".format(y_preds))
        # print("y_test:{}".format(y_test.values))
        error = 0;

        #scores = cross_val_score(model, X, Y, cv=cv, scoring=metrics)
        n_correct = sum(y_preds == y_test)
        acc = float(n_correct) / len(y_preds)
        metrics_num = acc

        if metrics_num > max_metrics:
            max_metrics = metrics_num
            best_comb = comb
        progress_str = "{} / {}, best comb: {}, best score: {}".format(idx + 1, len(combinations), best_comb, max_metrics)
        sys.stdout.write('\r' + progress_str)

    print("best params:{}".format(best_comb))
    print("max {}: {}".format(metrics, max_metrics))
    return best_comb

In [140]:
dt_param_grid = { "criterion" : ['gini', 'entropy'],
                  "splitter" : ["best", "random"],
                  "max_depth": [2, 3, 4, 5, 7, None],
                  "max_features": [None, 1, 2, 5, 10],
                  "min_samples_split": [10, 11, 12, 15, 20, 50],
                  "min_samples_leaf": [1, 2, 3, 5, 8, 10, 15, 20],
                  #"bootstrap": [True, False]
                 }
dt_best_comb = GridSearchWithVal(DecisionTreeClassifier, dt_param_grid)

5760 combinations in total. Metric: accuracy


101 / 5760, best comb: {'criterion': 'gini', 'splitter': 'best', 'max_depth': 2, 'max_features': None, 'min_samples_split': 10, 'min_samples_leaf': 5}, best score: 0.6217948717948718

157 / 5760, best comb: {'criterion': 'gini', 'splitter': 'best', 'max_depth': 2, 'max_features': 5, 'min_samples_split': 11, 'min_samples_leaf': 5}, best score: 0.6282051282051282

208 / 5760, best comb: {'criterion': 'gini', 'splitter': 'best', 'max_depth': 2, 'max_features': 5, 'min_samples_split': 11, 'min_samples_leaf': 5}, best score: 0.6282051282051282

257 / 5760, best comb: {'criterion': 'gini', 'splitter': 'best', 'max_depth': 2, 'max_features': 5, 'min_samples_split': 11, 'min_samples_leaf': 5}, best score: 0.6282051282051282

5760 / 5760, best comb: {'criterion': 'gini', 'splitter': 'best', 'max_depth': 2, 'max_features': 5, 'min_samples_split': 11, 'min_samples_leaf': 5}, best score: 0.6282051282051282best params:{'criterion': 'gini', 'splitter': 'best', 'max_depth': 2, 'max_features': 5, 'min_samples_split': 11, 'min_samples_leaf': 5}
max accuracy: 0.6282051282051282


In [ ]:
rf_param_grid = {"n_estimators" : [10, 50, 100, 500, 1000, 1500],
                  "criterion" : ["gini", "entropy"],
                  "max_depth": [2, 3, 4, 5, 7, None],
                  "max_features": [0.1, 0.2, 0.5, 0.8, None],
                  "min_samples_split": [2, 3, 4, 5],
                  "min_samples_leaf": [1, 3, 5, 10],
                  "bootstrap": [True, False]
                 }
rf_best = GridSearchWithVal(RandomForestClassifier, rf_param_grid)

11520 combinations in total. Metric: accuracy
9212 / 11520, best comb: {'n_estimators': 10, 'criterion': 'gini', 'max_depth': 2, 'max_features': 0.5, 'min_samples_split': 2, 'min_samples_leaf': 3, 'bootstrap': True}, best score: 0.6282051282051282